In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/faers_serious.csv')

def compute_prr(drug, reaction, df):
    a = len(df[(df['drugname']==drug) & (df['reaction']==reaction)])
    b = len(df[(df['drugname']==drug) & (df['reaction']!=reaction)])
    c = len(df[(df['drugname']!=drug) & (df['reaction']==reaction)])
    d = len(df[(df['drugname']!=drug) & (df['reaction']!=reaction)])
    
    if min(a,b,c,d) == 0:
        return None, None, None, a
    
    prr = (a/(a+b)) / (c/(c+d))
    
    # Confidence interval (UPGRADE)
    se = np.sqrt((1/a) - (1/(a+b)) + (1/c) - (1/(c+d)))
    lower = np.exp(np.log(prr) - 1.96*se)
    upper = np.exp(np.log(prr) + 1.96*se)
    
    return prr, lower, upper, a


In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/faers_serious.csv', usecols=['drugname', 'reaction'])

# Count only observed drug-reaction pairs (sparse), instead of all matrix combinations
pair_counts = (
    df.groupby(['drugname', 'reaction'])
      .size()
      .rename('a')
      .reset_index()
)

# Keep pairs with at least 3 cases (same filter as before)
pair_counts = pair_counts[pair_counts['a'] >= 3].copy()

# Marginal totals
drug_totals = df['drugname'].value_counts()
reaction_totals = df['reaction'].value_counts()
N = len(df)

# Lookup marginals for each observed pair
pair_counts['drug_total'] = pair_counts['drugname'].map(drug_totals)
pair_counts['reaction_total'] = pair_counts['reaction'].map(reaction_totals)

# Contingency cells
pair_counts['b'] = pair_counts['drug_total'] - pair_counts['a']
pair_counts['c'] = pair_counts['reaction_total'] - pair_counts['a']
pair_counts['d'] = N - (pair_counts['a'] + pair_counts['b'] + pair_counts['c'])

# Avoid invalid divisions
valid = (pair_counts[['a', 'b', 'c', 'd']] > 0).all(axis=1)
calc = pair_counts.loc[valid].copy()

# PRR and confidence intervals
calc['prr'] = (calc['a'] / (calc['a'] + calc['b'])) / (calc['c'] / (calc['c'] + calc['d']))
calc['se'] = np.sqrt((1 / calc['a']) - (1 / (calc['a'] + calc['b'])) + (1 / calc['c']) - (1 / (calc['c'] + calc['d'])))
calc['lower_ci'] = np.exp(np.log(calc['prr']) - 1.96 * calc['se'])
calc['upper_ci'] = np.exp(np.log(calc['prr']) + 1.96 * calc['se'])

signals = calc.loc[(calc['prr'] > 2) & (calc['lower_ci'] > 1), ['drugname', 'reaction', 'a', 'prr', 'lower_ci', 'upper_ci']].copy()
signals = signals.rename(columns={'drugname': 'drug', 'a': 'cases'})
signals['prr'] = signals['prr'].round(2)
signals['lower_ci'] = signals['lower_ci'].round(2)
signals['upper_ci'] = signals['upper_ci'].round(2)

# Backward-compatible with your original line style
results = signals.to_dict(orient='records')
signals = pd.DataFrame(results)

In [10]:
def classify(row):
    if row['prr'] > 3 and row['cases'] >= 10:
        return 'High'
    elif row['prr'] > 2:
        return 'Medium'
    return 'Low'

signals['priority'] = signals.apply(classify, axis=1)

In [11]:
signals.sort_values(by='prr', ascending=False)\
       .to_csv('../outputs/signals/flagged_signals.csv', index=False)